En este archivo Notebook realizo diferentes implementaciones del proyecto para poder ejecutarlas por bloques y obtener un resultado visual

In [13]:
# Añadimos todas las librerias necesarias
import oracledb as oracledb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Creación de la conexión a la base de datos

In [ ]:
# Crear el DSN usando la IP remota
dsn = oracledb.makedsn("afrodita.lcc.uma.es", 1521, sid="APOLO")

# Conectarse
conn = oracledb.connect(user="tfm_puertas", password="JCGRmlbEsc", dsn=dsn)

# Ejecutar consulta
cur = conn.cursor()
cur.execute("select distinct nombreasignatura from v_calificaciones")
for row in cur:
    print(row)

cur.close()

('Bases de Datos',)
('Estructura de Datos',)
('Ingeniería y Ciencia de Datos II',)
('Industrialización y Despliegue de Sistemas loT',)
('Planificación de Proyectos y Análisis de Riesgos',)
('Control Automático',)
('Programación de Videojuegos',)
('Diseño de Sistemas Operativos',)
('Bioquímica Estructural',)
('Física II',)
('Laboratorio de Computación Científica',)
('Administración de Bases de Datos',)
('Seguridad en Sistemas Industriales y Ciberfísicos',)
('Visión por Computador',)
('Sistemas de Información Empresarial',)
('Arquitecturas Paralelas',)
('Programación Distribuida',)
('Ingeniería del Software Avanzada',)
('Álgebra Lineal y Geometría',)
('Análisis Matemático III',)
('Telemedicina',)
('Inferencia Estadística',)
('Geometría Diferencial Global de Superficies',)
('Teoría de Autómatas y Lenguajes Formales',)
('Desarrollo de Aplicaciones en la Nube',)
('Sistemas de Información para Internet',)
('Diseño y Evaluación de Infraestructuras Informáticas',)
('Cognición y Comunicación en

Función que calcula la nota media ponderada.

Nota media ponderada: Media de todas las asignaturas aprobadas por el número de créditos que valen, dividido entre el total de créditos de la titulación

In [ ]:
def calcular_media_ponderada(conn, codigo_alumno, curso_max):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT 
                AVG(TO_NUMBER(NUM_CALIFICACIÓN)) AS media_aprobadas,
                COUNT(*) AS asignaturas_aprobadas
            FROM v_calificaciones
            WHERE CODIGOALUM = :codigo
            AND CALIFICACIÓN NOT IN ('NO PRESENTADO', 'SUSPENSO')
            AND TO_NUMBER(SUBSTR(CURSOACADÉMICO, 1, 4)) < TO_NUMBER(SUBSTR(:curso_inicio , 1, 4))
        """, codigo=codigo_alumno, curso_inicio=curso_max[:4])

        resultado = cur.fetchone() #Como esperamos solo un resultado, uso fetchone
        media, aprobadas = resultado

        if media is None or aprobadas == 0:
            return None  # Evitamos división por cero o falta de datos

        # Fórmula: (media * aprobadas * 6) / 240. NOTA: Supongo que todas las asignaturas valen 6 créditos ya que no tenemos información de cada una
        ponderada = (media * aprobadas * 6) / 240
        return round(ponderada, 2)
    
calcular_media_ponderada(conn, '0208F18506E41D3F29A4CAAD842FD0FA','2020-21')


1.57

In [33]:
def obtener_alumnos_matriculados(conn, nombre_asignatura):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT CODIGOALUM, CURSOACADÉMICO
            FROM v_calificaciones
            WHERE NOMBREASIGNATURA = :asignatura
        """, asignatura=nombre_asignatura)

        return cur.fetchall()
        
obtener_alumnos_matriculados(conn, 'Gestión Inteligente de la Información')


[('63959AA92F1209959086EF9705ACC5C9', '2018-19'),
 ('D3BEB47C85B5D63DD175EAFCFC89AA95', '2019-20'),
 ('180BE1AEABF5246B34EA5C48CDB4EA4A', '2019-20'),
 ('F9EE4394854A8F0688F82D8D850DE2BA', '2020-21'),
 ('2588945FFAD322856DD46C71D0974D25', '2020-21'),
 ('9FCD3D302D4B1F92D60C6EA52A4C55C2', '2021-22'),
 ('30D1F3A0A41167D565313EFC43687239', '2021-22'),
 ('E276D326633FC0BD66FDE06905B8FF31', '2018-19'),
 ('9C4267569C9FF29620C3084FAB122899', '2019-20'),
 ('CCF0F1035A0AACEA24AF588FA21E2D7B', '2018-19'),
 ('D50DF598D79919B965A82391A714DD54', '2018-19'),
 ('328DFF715F1AF124A64B2F69462E8883', '2019-20'),
 ('E0CEE1184B8A37EA15EA3DE72608EF58', '2019-20'),
 ('1397AAC9C27E7139804F672CDA335FDA', '2019-20'),
 ('2EECB97AF00DBC8F59E0E3732D5E3855', '2019-20'),
 ('09D3DA93C1195F9337C98092057E86BE', '2020-21'),
 ('9B2CEBEDA8DEF39ACB355C0109C4220D', '2020-21'),
 ('D84FE85097391DB0E66E43EB298ABAED', '2021-22'),
 ('90254D0B99DED4F38C897031CC34FFC4', '2021-22'),
 ('003F9854C10615AADCF46D3CE4B6F0AA', '2021-22'),


In [ ]:
def calcular_nota_corte(conn, nombre_asignatura):
    alumnos_con_curso = obtener_alumnos_matriculados(conn, nombre_asignatura)

    # 2. Agrupar alumnos por curso académico
    cursos = {}
    for codigo_alum, curso_acad in alumnos_con_curso:
        cursos.setdefault(curso_acad, []).append(codigo_alum)

    # 3. Para cada curso, calcular la nota de corte
    nota_corte_por_anio = {}

    for curso_acad, codigos_alumnos in cursos.items():
        medias = []
        for codigo_alum in codigos_alumnos:
            media = calcular_media_ponderada(conn, codigo_alum, curso_acad)
            if media is not None:
                print(media)
                medias.append(media)
        
        if medias:
            nota_corte_por_anio[curso_acad] = min(medias)

    return nota_corte_por_anio  # Dict: { "2018-19": 6.25, "2019-20": 5.8, ... }

calcular_nota_corte(conn, 'Visión por Computador')


{'2019-20': 0.17, '2020-21': 0.28, '2021-22': 0.3, '2022-23': 2.12}

In [41]:
def calcular_probabilidad_entrada(conn, nombre_asignatura, codigo_alumno):
    # Obtener las notas de corte por año
    notas_corte = calcular_nota_corte(conn, nombre_asignatura)

    if not notas_corte:
        return 0.0  # No hay base para calcular probabilidad

    total = 0
    supera = 0

    for curso_acad, nota_corte in notas_corte.items():
        media_alumno = calcular_media_ponderada(conn, codigo_alumno, curso_acad)
        
        if media_alumno is not None:
            total += 1
            if media_alumno > nota_corte:
                supera += 1

    if total == 0:
        return 0.0  # El alumno no tiene historial válido

    probabilidad = (supera / total) * 100
    return round(probabilidad, 2)

calcular_probabilidad_entrada(conn, 'Visión por Computador', '0208F18506E41D3F29A4CAAD842FD0FA')


100.0

In [10]:
#Cierre de conexión
conn.close()